# Polars intro

Polars is a blazingly fast DataFrame library for manipulating structured data. The core is written in Rust, and available for Python, R and NodeJS.

### Key features

- **Fast**: Written from scratch in Rust, designed close to the machine and without external
  dependencies.
- **I/O**: First class support for all common data storage layers: local, cloud storage & databases.
- **Intuitive API**: Write your queries the way they were intended. Polars, internally, will
  determine the most efficient way to execute using its query optimizer.
- **Out of Core**: The streaming API allows you to process your results without requiring all your
  data to be in memory at the same time.
- **Parallel**: Utilises the power of your machine by dividing the workload among the available CPU cores without any additional configuration.
- **Vectorized Query Engine**
- **GPU Support**: Optionally run queries on NVIDIA GPUs for maximum performance for in-memory workloads.
- **[Apache Arrow support](https://arrow.apache.org/)**: Polars can consume and produce Arrow data
  often with zero-copy operations. Note that Polars is not built on a Pyarrow/Arrow implementation.
  Instead, Polars has its own compute and buffer implementations.

### Potential annoyances for xarray users
- **2D DataFrames only**. No N-D data, but ways to still do nD things with `.group_by`
- **Very different syntax** from `numpy` and `pandas`

### Basically a mix of `pandas`, `SQL` and `dask` 
but with its own syntax. 

In [ ]:
import polars as pl
import datetime as dt

df = pl.read_csv("/Users/bandelol/Documents/code_local/data/iris.csv") # I/O like pandas (read and write, not open and to)
df.head()


In [ ]:
df["species"].unique()

In [ ]:
pl.lit([2, 4])

In [ ]:
expr = ((pl.col("sepal_length") + 32) * 3920).sqrt().pow(2)

In [ ]:
df.with_columns(custum_col=expr)

In [ ]:
q = (
    df
    .filter(pl.col("sepal_length") > 5) # filter using an Expression (equivalent-ish of boolean indexing)
    .group_by(pl.col("species"), maintain_order=True) # group by, super useful
    .agg(pl.all().mean()) # group wise aggregation, also using an Expression
)
q

In [ ]:
import polars as pl
import datetime as dt

df = pl.DataFrame(
    {
        "name": ["Alice Archer", "Ben Brown", "Chloe Cooper", "Daniel Donovan"],
        "birthdate": [
            dt.date(1997, 1, 10),
            dt.date(1985, 2, 15),
            dt.date(1983, 3, 22),
            dt.date(1981, 4, 30),
        ],
        "weight": [57.9, 72.5, 53.6, 83.1],  # (kg)
        "height": [1.56, 1.77, 1.65, 1.75],  # (m)
    }
)

df

# tricks for xarray users

In [ ]:
import xarray as xr
da = xr.open_mfdataset("/Users/bandelol/Documents/code_local/data/t2m/199?.nc")["t"]
da = da.sel(lon=slice(-80, 40), lat=slice(15, 80))

In [ ]:
da = da.compute()
da

In [ ]:
gb = da.groupby("time.dayofyear")
clim = gb.mean().compute()
anom = gb - clim
anom.compute()

In [ ]:
df = da.to_dataframe().reset_index(allow_duplicates=True)
df = pl.from_pandas(df)

In [ ]:
df

In [ ]:
df.group_by(["lon", "lat", pl.col("time").dt.ordinal_day().alias("dayofyear")], maintain_order=True).agg(
    pl.col("t").mean()
).filter(pl.col("dayofyear") < 366)

In [ ]:
clim = pl.col("t").mean()
anom = pl.col("t") - clim
dayofyear = pl.col("time").dt.ordinal_day().alias("dayofyear")
(
    df
    .group_by(["lon", "lat", dayofyear])
    .agg(clim=clim, anom=anom, time=pl.col("time"))
).explode("time", "anom")